# Context managers
## What's a context?

In [ ]:
from pathlib import Path
with Path("contexts.ipynb").open("r") as f:
    text = f.read()

# what it does:
# file.open() <...do something...>  file.close()
# 
# torch.no_grad(): switch Parameters to requires_grad = False, <...do something...> then turn it back again
#
# with concurrent.futures.ProcessPoolExecutor()

In [ ]:
class MyContext:
    
    def __init__(self, output):
        print("Creating context")
        self.output = output
    
    def __enter__(self):
        print(f"Entering context, returning {self.output}")
        return self.output
    
    def __exit__(self, *args):
        print(f"Exiting context: {args}")

In [ ]:
context = MyContext(12)

In [ ]:
with context as output:
    print(f"Within context, I got: {output}")

In [ ]:
with context as output:
    print("Let's make a mess")
    raise RuntimeError("Mess")

**With a context, never forget cleaning up your mess, even after an exception!**

## Simpler context managers with contextlib

In [ ]:
import contextlib

In [ ]:
@contextlib.contextmanager
def my_context(output):
    print(f"Entering context, yielding {output}")
    yield output
    print(f"Exiting context.")

In [ ]:
context = my_context(12)

In [ ]:
with context as output:
    print(f"Within context, I got: {output}")

In [ ]:
context = my_context(12)  # generators can only be used once
with context as output:
    raise RuntimeError("Something")

In [ ]:
@contextlib.contextmanager
def my_context(output):
    print(f"Entering context, yielding {output}")
    try:
        yield output
    except Exception as e:  # not necessary, "finally" is enough
        raise RuntimeError("I cleaned some mess") from e
    finally:
        print(f"Exiting context.")

In [ ]:
with my_context(12) as output:
    raise RuntimeError("Something")

In [ ]:
with my_context(12) as output:
    print("Hello")

## Example usage

Temporarily adding environment variables

In [ ]:
import os
import typing as tp

@contextlib.contextmanager
def environment_variables(**kwargs: tp.Any) -> tp.Iterator[None]:
    backup = {x: os.environ[x] for x in kwargs if x in os.environ}
    os.environ.update({x: str(y) for x, y in kwargs.items()})
    try:
        yield
    finally:
        for x in kwargs:
            del os.environ[x]
        os.environ.update(backup)

Saving to a buffer file: writing big files can be very slow, and you don't want someone to read it while it's being written

In [ ]:
import typing as tp
import contextlib
from pathlib import Path
import tempfile

@contextlib.contextmanager
def buffer_file(filepath: str | Path) -> tp.Iterator[Path]:
    filepath = Path(filepath).absolute()
    assert not filepath.exists()
    # Use a temporary directory to do the cleaning for us :)
    # dir parameter is not necessary, but writing on the same disk will make
    # moving faster.
    with tempfile.TemporaryDirectory(dir=filepath.parent) as tmp_folder:
        tmp_path = Path(tmp_folder) / filepath.name
        print(f"Entered context - Returning the temporary path: {tmp_path}")
        yield tmp_path
        print(f"Exiting context")
        if not tmp_path.exists():
            raise RuntimeError(f"You should have saved to {tmp_path}")
        tmp_path.rename(filepath)
    print(f"Temporary folder deleted: {not Path(tmp_folder).exists()}")
    print(f"Target file created: {filepath.exists()}")

In [ ]:
with buffer_file("hello.txt") as tmp:
    print(f"Getting a temporary file: {tmp}")


In [ ]:
[x for x in Path().iterdir() if x.is_dir()]  # folder has been deleted :)

In [ ]:
with buffer_file("hello.txt") as tmp:
    print(f"Getting a temporary file: {tmp}")
    tmp.write_text("world!")
Path("hello.txt").unlink()

More on contextlib https://docs.python.org/3/library/contextlib.html, eg:

    - suppress errors
    - redirect stdout/stderr
    - stack several contexts in one
    - write contexts that can be used as decorators